# 6.5. Bonus: IKEA produktu rasmošana — apraksti un cenas

[![Atvērt Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ValRCS/RTU_BDAA_Course_2026/blob/main/notebooks/lecture_06_web_scraping/06_05_bonus_ikea_products.ipynb)

Šajā bonus darba burtnīcā rasmosim **IKEA Latvija** produktu kategoriju un iegūsim produktu nosaukumus, īsos aprakstus, cenas, akcijas cenu informāciju un saites.

Izmantosim pašreizējo IKEA Latvija katalogu:
`https://www.ikea.com/lv/lv/cat/bernu-kreslini-18769/`

Plūsma:

**kategorijas URL → 1 HTTP pieprasījums → produktu kartītes → DataFrame → CSV**

Svarīgs princips: neatveram katra produkta lapu atsevišķi, jo vajadzīgie dati jau ir kategorijas lapā.


## Ko iemācīsimies

IKEA ir labs e-komercijas rasmošanas piemērs, jo vienā kartītē var būt pašreizējā cena, iepriekšējā cena, atlaides teksts, `IKEA Family` cena vai statusa etiķete.

Tāpēc cenu neņemam kā “pirmo skaitli no visas kartītes”. Vispirms atrodam konkrēto cenas HTML elementu un tikai pēc tam normalizējam tā tekstu.

HTML struktūra laika gaitā var mainīties. Ja selektori vairs nedarbojas, tie jāpārbauda pārlūka Developer Tools. Ja vietne atgriež `403` vai `429`, aizsardzību neapejam.


In [ ]:
import importlib.util
import subprocess
import sys

required = {"requests":"requests", "bs4":"beautifulsoup4", "pandas":"pandas", "lxml":"lxml"}
missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]

if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

print("Python:", sys.version.split()[0])
print("Vide:", "Google Colab" if "google.colab" in sys.modules else "lokāls Jupyter/VS Code")


## Importi un konfigurācija

Parastā lietošanā mainām tikai `CATEGORY_URL`. Šajā kategorijā ir neliels produktu skaits, tāpēc tā ir piemērota mācību piemēram.


In [ ]:
from pathlib import Path
import re
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup
from IPython.display import display

CATEGORY_URL = "https://www.ikea.com/lv/lv/cat/bernu-kreslini-18769/"
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; RTU-BDAA-teaching-example/1.0)"}
REQUEST_TIMEOUT = 20

OUTPUT_DIR = Path("data")
OUTPUT_FILE = OUTPUT_DIR / "ikea_bernu_kreslini.csv"

pd.set_option("display.max_colwidth", 120)
print(CATEGORY_URL)


## 1. `fetch_soup(url)` — vienīgā HTTP funkcija

HTTP pieprasījumu atdalām no HTML apstrādes. Pārējās funkcijas saņem jau lejupielādētu `BeautifulSoup` objektu vai konkrētu produkta kartīti.


In [ ]:
def fetch_soup(url: str) -> BeautifulSoup:
    response = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    return BeautifulSoup(response.text, "lxml")


## 2. Ielādējam kategoriju vienu reizi

IKEA produktu kartītēm pašlaik tiek izmantota klase `.plp-mastercard`. Ja kartītes netiek atrastas, visticamāk, mainījusies HTML struktūra.


In [ ]:
soup = fetch_soup(CATEGORY_URL)
product_cards = soup.select(".plp-mastercard")

print("Lapas virsraksts:", soup.title.get_text(" ", strip=True) if soup.title else "(nav title)")
print("Atrasto produktu kartīšu skaits:", len(product_cards))

if not product_cards:
    raise ValueError("Produktu kartītes netika atrastas. Pārbaudi selektoru '.plp-mastercard'.")


## 3. Pirms parsēšanas apskatām vienu kartīti

Šis ir svarīgs rasmošanas darba solis: vispirms izpētām reālos datus, tikai pēc tam rakstām parseri.


In [ ]:
print(product_cards[0].get_text(" ", strip=True)[:1000])


## 4. Palīgfunkcijas tekstam un cenām

`text_or_none()` droši apstrādā neobligātus HTML elementus.

`parse_eur()` atbalsta gan veselus skaitļus, gan Latvijas formātu ar decimālo komatu, piemēram, `12,99€`, kā arī tūkstošu atstarpes.


In [ ]:
def text_or_none(element) -> str | None:
    return element.get_text(" ", strip=True) if element is not None else None


PRICE_RE = re.compile(r"(?P<amount>\d[\d \u00a0]*(?:[,.]\d{1,2})?)")


def parse_eur(text: str | None) -> float | None:
    if not text:
        return None

    match = PRICE_RE.search(text)
    if not match:
        return None

    value = (
        match.group("amount")
        .replace(" ", "")
        .replace("\u00a0", "")
        .replace(",", ".")
    )
    return float(value)


In [ ]:
for example in ["12€", "12,99€ Cena 12,99€", "1 299€", None]:
    print(f"{example!r:25} -> {parse_eur(example)}")


## 5. Iepriekšējās vai standarta cenas atrašana

Akcijas kartītē var būt arī teksts `Iepriekšējā cena 79€` vai `Standarta cena: 79€`.

To meklējam atsevišķi, lai nesajauktu ar pašreizējo cenu vai tekstu, piemēram, `Ietaupi 20€`.


In [ ]:
PREVIOUS_PRICE_RE = re.compile(
    r"(?:Iepriekšējā cena|Standarta cena)\s*:?\s*"
    r"(?P<amount>\d[\d \u00a0]*(?:[,.]\d{1,2})?)\s*€",
    flags=re.IGNORECASE,
)


def parse_previous_price(card) -> float | None:
    match = PREVIOUS_PRICE_RE.search(card.get_text(" ", strip=True))
    if not match:
        return None

    value = (
        match.group("amount")
        .replace(" ", "")
        .replace("\u00a0", "")
        .replace(",", ".")
    )
    return float(value)


## 6. `parse_product_card(card, base_url)`

No vienas IKEA kartītes iegūstam:

- `name` — produkta sēriju/nosaukumu;
- `description` — īso aprakstu;
- `price_text` un `price_eur`;
- `previous_price_eur`, ja ir akcija;
- `status`, ja tāds ir;
- pilno produkta URL.

Galvenie CSS selektori ir `.plp-price-module__product-name`, `.plp-price-module__description`, `.plp-price-module__current-price` un `a.plp-product__image-link`.


In [ ]:
def parse_product_card(card, base_url: str) -> dict:
    name = text_or_none(card.select_one(".plp-price-module__product-name"))
    description = text_or_none(card.select_one(".plp-price-module__description"))

    price_element = card.select_one(".plp-price-module__current-price")
    if price_element is None:
        price_element = card.select_one(".plp-price__integer")

    price_text = (
        text_or_none(price_element)
        if price_element is not None
        else card.get_text(" ", strip=True)
    )

    link_element = card.select_one("a.plp-product__image-link[href]")
    if link_element is None:
        link_element = card.select_one('a[href*="/p/"]')

    url = urljoin(base_url, link_element["href"]) if link_element else None

    if name is None or description is None:
        heading_text = text_or_none(card.select_one("h3"))
        name = name or heading_text
        description = description or heading_text

    return {
        "name": name,
        "description": description,
        "price_text": price_text,
        "price_eur": parse_eur(price_text),
        "previous_price_eur": parse_previous_price(card),
        "status": text_or_none(card.select_one(".plp-product-badge")),
        "url": url,
    }


## 7. Visas kartītes → `DataFrame`

Šajā posmā internetu vairs neizmantojam. Apstrādājam jau lejupielādētās kartītes un izveidojam tabulu.


In [ ]:
products = [parse_product_card(card, CATEGORY_URL) for card in product_cards]
df = pd.DataFrame(products)

print("DataFrame forma:", df.shape)
display(df.head(10))

print("\nTrūkstošās vērtības:")
display(df.isna().sum().to_frame("missing"))


## 8. Datu kvalitātes pārbaude

Vismaz nosaukumam, aprakstam, pašreizējai cenai un URL vajadzētu būt aizpildītiem. `previous_price_eur` un `status` drīkst būt tukši.


In [ ]:
required_columns = ["name", "description", "price_eur", "url"]

assert len(df) > 0, "Nav iegūts neviens produkts."

for column in required_columns:
    print(f"{column:12} trūkst:", df[column].isna().sum())

if df[required_columns].isna().any().any():
    print("\nBrīdinājums: pārbaudi IKEA HTML struktūru un CSS selektorus.")
else:
    print("\nPamatlauki visiem produktiem ir iegūti veiksmīgi.")


## 9. Ātra cenu pārbaude

Pilna datu analīze būs 7. lekcijas tēma, bet jau šeit sakārtojam produktus pēc cenas un apskatām akcijas preces. Tas palīdz pamanīt parsēšanas kļūdas.


In [ ]:
display(
    df.sort_values("price_eur")
      [["name", "description", "price_eur", "previous_price_eur", "status"]]
      .head(10)
)

print("Produkti ar iepriekšējo / standarta cenu:")
display(
    df.loc[
        df["previous_price_eur"].notna(),
        ["name", "description", "price_eur", "previous_price_eur", "status"],
    ]
)


## 10. Saglabājam CSV


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")
print("Saglabāts:", OUTPUT_FILE.resolve())


## 11. Pilna plūsma vienā funkcijā

Kad eksperimentālais kods darbojas, to varam apkopot atkārtoti izmantojamā funkcijā: viens kategorijas URL iekšā, viens `DataFrame` ārā.


In [ ]:
def scrape_ikea_category(category_url: str) -> pd.DataFrame:
    category_soup = fetch_soup(category_url)
    cards = category_soup.select(".plp-mastercard")

    if not cards:
        raise ValueError("Produktu kartītes netika atrastas.")

    return pd.DataFrame([
        parse_product_card(card, category_url)
        for card in cards
    ])


## Kopsavilkums un papildu uzdevumi

Šajā bonus notebook izveidojām pilnu e-komercijas plūsmu:

**IKEA kategorija → requests → BeautifulSoup → CSS selektori → cenu parsēšana → DataFrame → CSV**

Papildu uzdevumi:

1. nomaini `CATEGORY_URL` pret citu IKEA kategoriju;
2. pievieno atsauksmju skaitu;
3. izveido `discount_eur = previous_price_eur - price_eur`;
4. izveido `discount_pct`;
5. saglabā rezultātu arī XLSX;
6. salīdzini divas IKEA kategorijas vienā `DataFrame`.
